# PadelPro Vision — gerar CSV com o `padel_analytics` (Colab)

Corre o pipeline (baseado no trabalho do João Silva, mas já a partir do **teu fork**)
numa GPU do Colab e exporta o `data.csv` (posições dos jogadores em metros) para depois
analisares no teu repo `padelpro-vision`.

**Independente do João**: o repo vem do teu fork e os pesos vêm só da tua Drive — nada aqui
lê ou escreve nos dele.

**Antes de começar:** menu `Runtime → Change runtime type → T4 GPU`.

Fluxo: GPU → clonar → instalar → pesos → escolher keypoints do campo → configurar → correr → descarregar CSV.

## 1. Confirmar GPU

In [ ]:
!nvidia-smi

## 2. Clonar o repo (o teu fork)

In [ ]:
%cd /content
![ -d padel_analytics ] || git clone https://github.com/Vasco-Rocha/padel_analytics.git padel_analytics
%cd /content/padel_analytics
!ls

## 3. Instalar dependências
Versões fixadas em 12/07/2026 (o `requirements.txt` original não fixava versões, e o Colab
partiu duas vezes por causa disso). O PyTorch já vem no Colab.

In [ ]:
!pip install -q ultralytics==8.4.92 supervision==0.29.1 opencv-python-headless==4.13.0.92 pims==0.7 plotly seaborn parse==1.22.1 matplotlib gdown==5.2.2
print('Dependências instaladas.')

## 4. Descarregar os pesos
Os pesos estão na Drive do João. Descarregamos a pasta e organizamos na estrutura que o `config.py` espera:
```
weights/players_detection/yolov8m.pt
weights/players_keypoints_detection/best.pt
weights/ball_detection/TrackNet_best.pt
weights/ball_detection/InpaintNet_best.pt
weights/court_keypoints_detection/best.pt
```

In [ ]:
import os, glob
from google.colab import drive
drive.mount('/content/drive')

# So' a TUA Drive -- nunca a do Joao. Ainda so' temos o yolov8m.pt migrado para aqui;
# os outros pesos (best.pt x2, TrackNet_best.pt, InpaintNet_best.pt) nao se usam no
# fluxo atual (ver HANDOFF_INDEPENDENCIA.md) e nao foram migrados.
DRIVE_PESOS = '/content/drive/MyDrive/PadelPro/pesos'
os.makedirs('/content/padel_analytics/_weights_dl', exist_ok=True)
encontrados = glob.glob(os.path.join(DRIVE_PESOS, '**', '*.pt'), recursive=True)
for p in encontrados:
    shutil_dst = f'/content/padel_analytics/_weights_dl/{os.path.basename(p)}'
    import shutil; shutil.copy(p, shutil_dst)
    print('copiado da tua Drive ->', shutil_dst)
if not encontrados:
    print(f'Nada encontrado em {DRIVE_PESOS}.')
print('\nSe faltar algum peso (best.pt, TrackNet_best.pt, InpaintNet_best.pt), sobe-o a mao')
print('para a tua Drive em MyDrive/PadelPro/pesos/ -- nao ha fallback para a Drive do Joao.')

In [ ]:
# Arruma os ficheiros .pt na estrutura esperada (best-effort por nome).
import os, shutil, glob
DST = {
    'yolov8m.pt': 'weights/players_detection/yolov8m.pt',
    'TrackNet_best.pt': 'weights/ball_detection/TrackNet_best.pt',
    'InpaintNet_best.pt': 'weights/ball_detection/InpaintNet_best.pt',
}
found = {os.path.basename(p): p for p in glob.glob('/content/padel_analytics/_weights_dl/**/*.pt', recursive=True)}
for name, dst in DST.items():
    if name in found:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(found[name], dst); print('OK ->', dst)
    else:
        print('FALTA:', name, '(verifica a lista acima e copia à mão)')
print('\nNOTA: há dois ficheiros best.pt (pose dos jogadores e keypoints do campo).')
print('Confirma na lista acima qual é qual e copia para:')
print('  weights/players_keypoints_detection/best.pt')
print('  weights/court_keypoints_detection/best.pt')

## 5. Escolher o vídeo
Para um primeiro teste usa o clip que vem no repo (`examples/videos/rally.mp4`).
Para usar um vídeo teu, descomenta o bloco de upload.

In [ ]:
VIDEO_PATH = '/content/padel_analytics/examples/videos/rally.mp4'

# --- para usar o teu próprio clip, descomenta: ---
# from google.colab import files
# up = files.upload()
# VIDEO_PATH = '/content/padel_analytics/' + list(up.keys())[0]

print('Vídeo:', VIDEO_PATH)

## 6. Selecionar os 12 keypoints do campo (clicavel)
Corre a celula: aparece o 1.o frame. **Clica nos 12 keypoints por ordem k1->k12** (a etiqueta em cima diz qual e o proximo). Cada clique marca o ponto; 'Anular ultimo' corrige erros. Ao 12.o clique grava sozinho.
```
        k11--------------------k12
        |                       |
        k8-----------k9--------k10
        |            |          |
        k6----------------------k7   (rede)
        |            |          |
        k3-----------k4---------k5
        |                       |
        k1----------------------k2
```
Ordem: k1,k2 = cantos de fundo de baixo; k3,k4,k5 = linha de servico de baixo (esq/centro/dir); k6,k7 = pontas da rede; k8,k9,k10 = linha de servico de cima; k11,k12 = cantos de fundo de cima.

In [ ]:
import cv2, base64, json, os
from google.colab import output

cap = cv2.VideoCapture(VIDEO_PATH); ok, frame = cap.read(); cap.release()
assert ok, 'Nao consegui ler o video'
h, w = frame.shape[:2]
_, buf = cv2.imencode('.png', frame)
b64 = base64.b64encode(buf).decode()

js = '''
(async function() {
  const N = 12;
  const labels = ['k1','k2','k3','k4','k5','k6','k7','k8','k9','k10','k11','k12'];
  const img = new Image(); img.src = 'data:image/png;base64,%B64%'; await img.decode();
  const W=%W%, H=%H%, scale = Math.min(1, 1100/W);
  const c = document.createElement('canvas');
  c.width=W*scale; c.height=H*scale; c.style.cursor='crosshair'; c.style.border='1px solid #888';
  const ctx=c.getContext('2d');
  const info=document.createElement('div'); info.style.cssText='color:#0f0;font:14px monospace;margin:6px';
  const btn=document.createElement('button'); btn.textContent='Anular ultimo'; btn.style.margin='6px';
  document.body.append(info, c, btn);
  const pts=[];
  function draw(){ ctx.drawImage(img,0,0,c.width,c.height); ctx.font='16px sans-serif';
    pts.forEach((p,i)=>{ ctx.fillStyle='red'; ctx.beginPath(); ctx.arc(p.cx,p.cy,4,0,6.3); ctx.fill();
      ctx.fillStyle='yellow'; ctx.fillText(labels[i],p.cx+6,p.cy-6); }); }
  function st(){ info.textContent='Clica '+(labels[pts.length]||'(completo)')+' - '+pts.length+'/'+N; }
  draw(); st();
  return await new Promise(res=>{
    c.onclick=e=>{ if(pts.length>=N) return; const r=c.getBoundingClientRect();
      const cx=e.clientX-r.left, cy=e.clientY-r.top; pts.push({cx,cy,x:cx/scale,y:cy/scale});
      draw(); st(); if(pts.length===N) res(JSON.stringify(pts.map(p=>[p.x,p.y]))); };
    btn.onclick=()=>{ pts.pop(); draw(); st(); };
  });
})()
'''.replace('%B64%', b64).replace('%W%', str(w)).replace('%H%', str(h))

result = output.eval_js(js)
SELECTED_KEYPOINTS = json.loads(result)
os.makedirs('/content/padel_analytics/cache', exist_ok=True)
with open('/content/padel_analytics/cache/fixed_keypoints_detection.json', 'w') as f:
    json.dump([[float(x), float(y)] for x, y in SELECTED_KEYPOINTS], f)
print('Guardado:', SELECTED_KEYPOINTS)

In [ ]:
# FALLBACK (so se o modo clicavel acima falhar): preenche a mao e corre esta celula.
import json, os
# Preenche os 12 pares (x, y) pela ordem k1..k12, lidos da imagem acima.
SELECTED_KEYPOINTS = [
    # (x1, y1),   # k1
    # (x2, y2),   # k2
    # ... até k12
]
assert len(SELECTED_KEYPOINTS) == 12, 'Precisas de exatamente 12 keypoints (k1..k12).'
os.makedirs('/content/padel_analytics/cache', exist_ok=True)
with open('/content/padel_analytics/cache/fixed_keypoints_detection.json', 'w') as f:
    json.dump([[float(x), float(y)] for x, y in SELECTED_KEYPOINTS], f)
print('Keypoints guardados em cache/fixed_keypoints_detection.json')

## 7. Configurar o `config.py`
Reescrevemos o `config.py` para: usar o vídeo escolhido, carregar os keypoints do JSON (sem janela),
limitar a poucos frames no primeiro teste, ativar a recolha de dados e usar cache (`LOAD=None` no 1.º run).

In [ ]:
MAX_FRAMES = 300   # ~10s a 30fps para o 1.º teste; põe None para o vídeo todo
config = f'''""" Config gerado pelo notebook PadelPro Vision """
INPUT_VIDEO_PATH = "{VIDEO_PATH}"
OUTPUT_VIDEO_PATH = "results.mp4"
COLLECT_DATA = True
COLLECT_DATA_PATH = "data.csv"
MAX_FRAMES = {MAX_FRAMES}

FIXED_COURT_KEYPOINTS_LOAD_PATH = "./cache/fixed_keypoints_detection.json"
FIXED_COURT_KEYPOINTS_SAVE_PATH = None

PLAYERS_TRACKER_MODEL = "./weights/players_detection/yolov8m.pt"
PLAYERS_TRACKER_BATCH_SIZE = 8
PLAYERS_TRACKER_ANNOTATOR = "rectangle_bounding_box"
PLAYERS_TRACKER_LOAD_PATH = None
PLAYERS_TRACKER_SAVE_PATH = "./cache/players_detections.json"

PLAYERS_KEYPOINTS_TRACKER_MODEL = "./weights/players_keypoints_detection/best.pt"
PLAYERS_KEYPOINTS_TRACKER_TRAIN_IMAGE_SIZE = 1280
PLAYERS_KEYPOINTS_TRACKER_BATCH_SIZE = 8
PLAYERS_KEYPOINTS_TRACKER_LOAD_PATH = None
PLAYERS_KEYPOINTS_TRACKER_SAVE_PATH = "./cache/players_keypoints_detections.json"

BALL_TRACKER_MODEL = "./weights/ball_detection/TrackNet_best.pt"
BALL_TRACKER_INPAINT_MODEL = "./weights/ball_detection/InpaintNet_best.pt"
BALL_TRACKER_BATCH_SIZE = 8
BALL_TRACKER_MEDIAN_MAX_SAMPLE_NUM = 400
BALL_TRACKER_LOAD_PATH = None
BALL_TRACKER_SAVE_PATH = "./cache/ball_detections.json"

KEYPOINTS_TRACKER_MODEL = "./weights/court_keypoints_detection/best.pt"
KEYPOINTS_TRACKER_BATCH_SIZE = 8
KEYPOINTS_TRACKER_MODEL_TYPE = "yolo"
KEYPOINTS_TRACKER_LOAD_PATH = None
KEYPOINTS_TRACKER_SAVE_PATH = None
'''
with open('/content/padel_analytics/config.py', 'w') as f:
    f.write(config)
print('config.py escrito.')

## 8. Correr o pipeline
Gera o `data.csv`. No 1.º teste pode demorar alguns minutos (faz inferência dos 4 modelos).

In [ ]:
%cd /content/padel_analytics
!python main.py

## 9. Ver as bounding boxes (confirmar visualmente)
Mostra 4 frames do video anotado (`results.mp4`) com as caixas dos jogadores, bola e campo.
Se as caixas estiverem em cima dos jogadores, a detecao esta a funcionar.

In [ ]:
import cv2, matplotlib.pyplot as plt
RESULT = '/content/padel_analytics/results.mp4'
cap = cv2.VideoCapture(RESULT)
n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
assert n > 0, 'results.mp4 vazio/nao encontrado — confirma que o main.py correu sem erros'
idxs = [int(n * f) for f in (0.2, 0.4, 0.6, 0.8)]
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, i in zip(axes.ravel(), idxs):
    cap.set(cv2.CAP_PROP_POS_FRAMES, i)
    ok, fr = cap.read()
    if ok:
        ax.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); ax.set_title(f'frame {i}'); ax.axis('off')
cap.release(); plt.tight_layout(); plt.show()

In [ ]:
# (opcional) descarregar o video anotado completo para veres em movimento
from google.colab import files
files.download('/content/padel_analytics/results.mp4')

## 11. Ver e descarregar o CSV

In [ ]:
import pandas as pd
df = pd.read_csv('/content/padel_analytics/data.csv')
print('linhas:', len(df), '| colunas:', len(df.columns))
print([c for c in df.columns if c.startswith('player') and ('_x' in c or '_y' in c or 'Vnorm1' in c)][:12])
df.head()

In [ ]:
from google.colab import files
files.download('/content/padel_analytics/data.csv')
# (opcional) descarregar também as deteções da bola para o módulo de pancadas/tempo útil:
# files.download('/content/padel_analytics/cache/ball_detections.json')

## 12. Próximo passo (no teu Mac)
Com o `data.csv` descarregado, no repo `padelpro-vision`:
```bash
python -m padelpro.cli analyze --players data.csv --fps 30 --out analysis.json
```
Ajusta os nomes das colunas em `config.yaml` se necessario.